In [1]:
from langchain_ollama import ChatOllama
import requests
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate
import json
import re

In [2]:
search_tool = DuckDuckGoSearchRun(
    description = """When ever you dont know or confused on a particular topic, use this
    tool to get infomation about unknown/confused topics."""
)

In [3]:
@tool
def get_weather_data(city: str) -> str:
    """Fetches real-time weather for a city. Input should be a plain city name."""
    # Clean the input: remove "city:", quotes, extra spaces
    city = re.sub(r'^city:\s*', '', city, flags=re.IGNORECASE)
    city = city.strip('"\'')
    city = city.strip()
    
    url = f'https://api.weatherstack.com/current?access_key=22105a6d0b1ed0d864a741e198b9d199&query={city}'
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if 'location' not in data or 'current' not in data:
            return f"Error: No weather data found for '{city}'. Please verify the city name."
        
        # Optional: check location match
        returned_city = data['location']['name'].lower()
        requested = city.lower()
        if requested not in returned_city and returned_city not in requested:
            return f"Location mismatch: API returned '{data['location']['name']}, {data['location']['country']}' for input '{city}'. Try a more specific city name."
        
        weather = {
            "city": data['location']['name'],
            "country": data['location']['country'],
            "temperature": data['current']['temperature'],
            "description": data['current']['weather_descriptions'][0],
            "humidity": data['current']['humidity'],
            "wind_speed": data['current']['wind_speed']
        }
        return json.dumps(weather)
    except Exception as e:
        return f"Error fetching weather: {str(e)}"

In [4]:
get_weather_data.invoke("dhaka")

'{"city": "Dhaka", "country": "Bangladesh", "temperature": 37, "description": "Thundery outbreaks in nearby", "humidity": 45, "wind_speed": 22}'

In [5]:
model = ChatOllama(
    model = "qwen2.5:1.5b"
)

In [6]:
model_with_tools = model.bind_tools([search_tool, get_weather_data])

In [7]:
# prompt = PromptTemplate.from_template("""
# Answer the following questions as best you can. You have access to the following tools:

# {tools}

# Use the following format:

# Question: the input question you must answer
# Thought: you should always think about what to do
# Action: the action to take, should be one of [{tool_names}]
# Action Input: the input to the action
# Observation: the result of the action
# ... (repeat Thought/Action/Action Input/Observation as needed)
# Thought: I now know the final answer
# Final Answer: the final answer to the original input question

# Begin!

# Question: {input}
# Thought: {agent_scratchpad}
# """)

In [8]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["LANGSMITH_API_KEY"] = os.getenv('LANGCHAIN_API')

In [15]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react" , dangerously_pull_public_prompt=True)
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [16]:
agent = create_react_agent(
    llm = model_with_tools, 
    tools = [search_tool , get_weather_data],
    prompt = prompt
)

In [17]:
agent_executer = AgentExecutor(
    agent = agent,
    tools = [search_tool , get_weather_data],
    verbose = True
)

In [23]:
response = agent_executer.invoke({
    "input": "weather condition in dhaka today"
})



> Entering new AgentExecutor chain...
I should use the get_weather_data function to find out the current weather conditions for Dhaka.
Action: get_weather_data
Action Input: city: "dhaka"{"city": "Dhaka", "country": "Bangladesh", "temperature": 37, "description": "Thundery outbreaks in nearby", "humidity": 45, "wind_speed": 22}I now know the final answer
Final Answer: Today's weather conditions in Dhaka are as follows:

- Temperature: 37°C (in Celsius)
- Description: Thundery outbreaks near 
- Humidity: 45%
- Wind speed: 22 km/h

The observation shows that there may be thunderstorms nearby.

> Finished chain.


In [24]:
print(response['output'])

Today's weather conditions in Dhaka are as follows:

- Temperature: 37°C (in Celsius)
- Description: Thundery outbreaks near 
- Humidity: 45%
- Wind speed: 22 km/h

The observation shows that there may be thunderstorms nearby.
